# UTS Data Acquisition & Management — Akomodasi Pariwisata Kecamatan Ubud

**Nama:** Ni Putu Riska Pramesti Cahya Dewi  
**NIM:** 2501010315  
**Prodi:** Informatika — INSTIKI  
**Topik:** Direktori akomodasi Kecamatan Ubud, Kabupaten Gianyar.

**Catatan periode:** Publikasi BPS terbit tahun 2017, tetapi direktori yang dianalisis adalah kondisi/data tahun 2015. Data ini bersifat historis, bukan daftar akomodasi aktif saat ini. Notebook akan mengambil data cuaca historis Januari 2025 dari Open-Meteo dan melakukan scraping satu tabel statistik BPS Provinsi Bali jika robots.txt mengizinkan.

## 0. Persiapan, folder, dan konfigurasi request

In [ ]:
# Import pustaka untuk membaca data, mengakses API, memeriksa robots.txt, scraping, dan metadata.
from pathlib import Path
from datetime import datetime
from io import StringIO
import os, re, time, json, hashlib
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser
from IPython.display import display

# Cari folder proyek secara otomatis agar notebook bisa dijalankan setelah ZIP diekstrak di Colab.
NAMA_FOLDER = 'UTS_DAM_UBUD_RISKA_2501010315_FINAL'
calon = [Path('/content')/NAMA_FOLDER, Path.cwd()/NAMA_FOLDER, Path.cwd()]
BASE_DIR = next((p for p in calon if (p/'data'/'interim'/'akomodasi_ubud_bps_2015_ekstraksi.csv').exists()), None)
if BASE_DIR is None:
    raise FileNotFoundError('Folder proyek belum ditemukan. Ekstrak ZIP ke /content terlebih dahulu, lalu jalankan ulang notebook.')
RAW_DIR = BASE_DIR/'data'/'raw'; INTERIM_DIR = BASE_DIR/'data'/'interim'
PROCESSED_DIR = BASE_DIR/'data'/'processed'; META_DIR = BASE_DIR/'metadata'; DOCS_DIR = BASE_DIR/'docs'
for folder in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR, META_DIR, DOCS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# Email kampus harus diisi sendiri; alamat email individual tidak dapat ditebak dari email kontak umum kampus.
EMAIL_KAMPUS = os.getenv('EMAIL_KAMPUS', '').strip()
if not EMAIL_KAMPUS or '@' not in EMAIL_KAMPUS:
    EMAIL_KAMPUS = input('Masukkan email institusi mahasiswa INSTIKI (bukan email humas): ').strip()
if '@' not in EMAIL_KAMPUS:
    raise ValueError('Email belum valid. Masukkan email institusi mahasiswa yang benar sesuai instruksi dosen.')
USER_AGENT = f'UTS-DAM-2501010315 ({EMAIL_KAMPUS})'
TIMEOUT = 30
JEDA_DETIK = 2

# Fungsi request menggunakan User-Agent, timeout, jeda, pemeriksaan status, dan retry untuk error sementara.
def request_get(url, params=None, max_attempts=4):
    headers = {'User-Agent': USER_AGENT, 'Accept': 'text/html,application/json;q=0.9,*/*;q=0.8'}
    for attempt in range(1, max_attempts + 1):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=TIMEOUT)
            time.sleep(JEDA_DETIK)
            if response.status_code in [429, 500, 502, 503, 504] and attempt < max_attempts:
                retry_after = response.headers.get('Retry-After')
                tunggu = int(retry_after) if retry_after and retry_after.isdigit() else JEDA_DETIK * attempt
                time.sleep(tunggu)
                continue
            response.raise_for_status()
            return response
        except requests.RequestException:
            if attempt == max_attempts:
                raise
            time.sleep(JEDA_DETIK * attempt)

print('Folder proyek:', BASE_DIR)
print('Email kampus berhasil dimasukkan ke User-Agent (tidak dicetak atau disimpan ke provenance).')


## Soal 1 — Membaca dan mengidentifikasi dataset

In [ ]:
# Baca dataset turunan hasil ekstraksi tabel BPS; PDF asli tetap tidak diubah di data/raw.
DATASET = INTERIM_DIR/'akomodasi_ubud_bps_2015_ekstraksi.csv'
df_akomodasi = pd.read_csv(DATASET, encoding='utf-8-sig', dtype={
    'no_sumber':'Int64', 'desa':'string', 'nama_akomodasi':'string', 'alamat':'string',
    'klasifikasi':'string', 'kecamatan':'string', 'tahun_data':'Int64'
})

# Hitung ukuran PDF sumber dan estimasi storage sesuai rumus soal.
pdf_path = RAW_DIR/'BPS5104-direktori_hotel_kabupaten_gianyar_2017.pdf'
ukuran_byte = pdf_path.stat().st_size
ukuran_mb = ukuran_byte / (1024**2)
estimasi_byte = ukuran_byte * 12 * 1.30
print('Jenis data: terstruktur (CSV)')
print('Jumlah baris:', len(df_akomodasi), '| Jumlah kolom:', len(df_akomodasi.columns))
print('Ukuran PDF sumber (MB):', round(ukuran_mb, 3))
print('Estimasi 1 tahun + cadangan 30% (MB):', round(estimasi_byte/(1024**2), 3))
print('Tipe data:'); display(df_akomodasi.dtypes.to_frame('tipe_data'))
print('Nilai kosong:'); display(df_akomodasi.isna().sum().to_frame('jumlah_kosong'))
print('Duplikasi penuh:', int(df_akomodasi.duplicated().sum()))
print('Klasifikasi akomodasi:'); display(df_akomodasi['klasifikasi'].value_counts().to_frame('jumlah'))
print('Desa:'); display(df_akomodasi['desa'].value_counts().to_frame('jumlah'))
display(df_akomodasi.head())
assert len(df_akomodasi) >= 100, 'Dataset harus memiliki minimal 100 baris.'
assert df_akomodasi.duplicated().sum() == 0, 'Periksa baris duplikat.' 

### Interpretasi Soal 1
- **Jenis data:** terstruktur, karena tersusun dalam baris dan kolom.
- **Unit observasi:** satu entri usaha akomodasi.
- **Jumlah:** 326 baris dan 7 kolom pada file ekstraksi yang disediakan.
- **Wilayah:** Kecamatan Ubud, Kabupaten Gianyar, Bali.
- **Waktu:** data direktori tahun 2015; publikasi BPS terbit 2017.
- **Variabel:** `no_sumber` (nomor urut, bukan ukuran), `desa`, `nama_akomodasi`, `alamat`, `klasifikasi`, `kecamatan`, `tahun_data`. Kolom telepon dari publikasi tidak dimasukkan ke dataset turunan.
- **Satuan:** jumlah akomodasi dihitung dalam unit usaha; suhu dalam °C, hujan dalam mm, dan kecepatan angin dalam km/jam pada data API.
- **Storage:** rumus sesuai instruksi dosen = ukuran PDF sumber × 12 bulan × 1,30. Asumsi ukuran sumber bulanan tetap; ini simulasi kapasitas, bukan pertumbuhan data aktual.

## Soal 2 — Lisensi, PII, quasi-identifier, dan Data Ethics Checklist

In [ ]:
# Baca checklist etika yang disiapkan dan tampilkan seluruh sembilan butir beserta bukti.
from openpyxl import load_workbook
checklist_path = META_DIR/'data_ethics_checklist.xlsx'
wb = load_workbook(checklist_path, data_only=True)
for row in wb['Checklist Etika'].iter_rows(values_only=True):
    print(row)

# Catat ringkasan kolom yang berpotensi sensitif untuk audit etika.
print('Kolom dataset turunan:', list(df_akomodasi.columns))
print('Kolom telepon ada di dataset turunan:', 'telepon' in [c.lower() for c in df_akomodasi.columns])

### Analisis lisensi dan etika
Publikasi BPS memuat larangan pengumuman, distribusi, komunikasi, atau penggandaan sebagian/seluruh isi untuk tujuan komersial tanpa izin tertulis BPS. Karena itu, akses unduh publik **tidak** boleh dianggap sebagai lisensi terbuka tanpa batas. Tugas ini menggunakan data untuk kepentingan akademik dan mencantumkan atribusi BPS.

PDF sumber memuat nama usaha, alamat, nomor telepon, dan klasifikasi. Nomor telepon dapat menjadi data kontak pribadi/pengelola; kolom tersebut tidak dimasukkan ke CSV analisis. Kombinasi desa dan alamat rinci berpotensi menjadi quasi-identifier lokasi. Checklist sembilan butir disimpan pada `metadata/data_ethics_checklist.xlsx` dan CSV pendamping.

## Soal 3 — Akuisisi API Open-Meteo (minimal 30 baris)

In [ ]:
# Ambil cuaca historis harian selama 31 hari untuk koordinat Ubud melalui Open-Meteo.
API_URL = 'https://archive-api.open-meteo.com/v1/archive'
params = {
    'latitude': -8.5069, 'longitude': 115.2625,
    'start_date': '2025-01-01', 'end_date': '2025-01-31',
    'daily': 'temperature_2m_mean,precipitation_sum,wind_speed_10m_max',
    'timezone': 'Asia/Makassar'
}
api_response = request_get(API_URL, params=params)
print('HTTP status API:', api_response.status_code)
raw_api = api_response.json()

# Simpan body respons JSON mentah tanpa serialisasi ulang agar isi sumber tetap apa adanya.
api_raw_path = RAW_DIR/'open_meteo_ubud_2025-01.json'
api_raw_path.write_text(api_response.text, encoding='utf-8')
daily = raw_api.get('daily', {})
if not daily or 'time' not in daily:
    raise ValueError('Respons API tidak memiliki data harian yang diharapkan: periksa pesan error Open-Meteo.')
df_cuaca = pd.DataFrame({
    'tanggal': daily['time'],
    'suhu_rata2_c': daily['temperature_2m_mean'],
    'curah_hujan_mm': daily['precipitation_sum'],
    'kecepatan_angin_maks_kmh': daily['wind_speed_10m_max']
})
df_cuaca['kecamatan'] = 'Ubud'
assert len(df_cuaca) >= 30, 'Data API kurang dari 30 baris.'
df_cuaca.to_csv(INTERIM_DIR/'cuaca_harian_ubud_2025-01.csv', index=False, encoding='utf-8-sig')
print('Jumlah baris API:', len(df_cuaca)); display(df_cuaca.head())
print('Satuan: suhu °C; curah hujan mm; kecepatan angin km/jam.')


**Penjelasan API:** Portal direktori BPS yang menjadi sumber utama tidak menyediakan endpoint API yang digunakan pada tugas ini. Oleh karena itu digunakan Open-Meteo Historical Weather API sebagai API publik yang relevan dengan konteks lokasi Ubud. Data API berisi cuaca harian Januari 2025, bukan data jumlah akomodasi. Dokumentasi Open-Meteo menyebut endpoint arsip historis dan lisensi CC BY 4.0; tetap cantumkan atribusi sumber API.

## Soal 4 — Pemeriksaan robots.txt dan scraping halaman statis BPS

In [ ]:
# Periksa robots.txt sebelum mengakses halaman tabel statistik BPS Provinsi Bali.SCRAPE_BASE = 'https://bali.bps.go.id'ROBOTS_URL = urljoin(SCRAPE_BASE, '/robots.txt')SCRAPE_URL = 'https://bali.bps.go.id/en/statistics-table/3/ZEdWTmFXVlpWMDFCUms5MFJuTTNabEIyVVZOSlVUMDkjMyM1MTAw/jumlah-akomodasi-kamar-dan-tempat-tidur-yang-tersedia-pada-hotel-nonbintang-dan-akomodasi-lainnya-menurut-kabupaten-kota-di-provinsi-bali.html'robots_path = RAW_DIR/'robots_bali_bps.txt'robots_response = Nonerobots_status = 'GAGAL: request robots.txt tidak berhasil'robots_text = ''allowed = False# Periksa robots.txt dengan User-Agent, timeout, dan jeda; status non-200 tidak menghentikan notebook.try:    headers = {'User-Agent': USER_AGENT, 'Accept': 'text/plain,*/*;q=0.8'}    robots_response = requests.get(ROBOTS_URL, headers=headers, timeout=TIMEOUT)    time.sleep(JEDA_DETIK)    robots_status = robots_response.status_code    robots_text = robots_response.text    robots_path.write_bytes(robots_response.content)    if robots_response.status_code == 200:        robot_parser = RobotFileParser()        robot_parser.parse(robots_text.splitlines())        allowed = robot_parser.can_fetch(USER_AGENT, SCRAPE_URL)    elif robots_response.status_code == 404:        # robots.txt tidak ditemukan; tetap perhatikan ketentuan penggunaan portal.        allowed = True    else:        # Fail-closed: jangan scraping jika aturan tidak dapat diverifikasi.        allowed = Falseexcept requests.RequestException as exc:    print('Gagal mengambil robots.txt:', str(exc))    allowed = Falseprint('URL robots:', ROBOTS_URL, '| HTTP:', robots_status)print('URL target:', SCRAPE_URL)print('Diizinkan robots.txt:', allowed)# Scraping hanya dilakukan jika robots.txt berhasil diperiksa dan mengizinkan akses.tables = []html_raw_path = RAW_DIR/'halaman_bps_akomodasi.html'judul_halaman = 'Tidak diakses karena robots.txt tidak mengizinkan atau tidak dapat diverifikasi'if robots_response is None or (robots_response.status_code not in [200, 404]):    status_scraping = f'DILEWATI: robots.txt tidak dapat diverifikasi ({robots_status})'elif not allowed:    status_scraping = 'DILEWATI: robots.txt tidak mengizinkan akses'else:    status_scraping = 'DIIZINKAN: menunggu pengambilan halaman'    html_response = request_get(SCRAPE_URL)    print('HTTP status halaman:', html_response.status_code)    html_raw_path.write_bytes(html_response.content)    soup = BeautifulSoup(html_response.text, 'html.parser')    judul_halaman = soup.title.get_text(' ', strip=True) if soup.title else 'Judul tidak ditemukan'    # Ambil tabel HTML statis; jangan membuat data pengganti jika tidak tersedia.    try:        tables = pd.read_html(StringIO(html_response.text))    except ValueError:        tables = []    if tables:        for idx, table in enumerate(tables, start=1):            table.to_csv(INTERIM_DIR/f'bps_bali_tabel_scraping_{idx}.csv', index=False, encoding='utf-8-sig')            print('Tabel', idx, '| bentuk:', table.shape)            display(table.head())        status_scraping = f'BERHASIL: {len(tables)} tabel HTML ditemukan'    else:        status_scraping = 'TIDAK LENGKAP: halaman dapat diakses tetapi tabel HTML statis tidak ditemukan'# Simpan bukti pemeriksaan dan status aktual.(DOCS_DIR/'bukti_scraping.txt').write_text(    f'Judul: {judul_halaman}\\nRobots URL: {ROBOTS_URL}\\nTarget URL: {SCRAPE_URL}\\n'    f'HTTP robots: {robots_status}\\nDiizinkan: {allowed}\\nStatus scraping: {status_scraping}\\n',    encoding='utf-8')print('Status akhir scraping:', status_scraping)

**Catatan tingkat data:** tabel BPS Provinsi Bali merupakan statistik agregat kabupaten/kota, sedangkan dataset utama berisi daftar usaha tingkat Kecamatan Ubud. Karena tingkat agregasinya berbeda, hasil scraping didokumentasikan sebagai sumber ketiga dan tidak dipaksakan untuk digabung ke setiap baris akomodasi. Jika halaman berubah atau tabel tidak tersedia, notebook akan berhenti dengan pesan error; jangan mengganti hasil dengan data buatan.

## Soal 5 — Penggabungan data dan validasi

In [ ]:
# Ringkas data cuaca harian menjadi satu observasi per kecamatan agar dapat digabungkan dengan data akomodasi.
df_cuaca_ringkas = pd.DataFrame([{
    'kecamatan': 'Ubud',
    'periode_cuaca': '2025-01',
    'suhu_rata2_bulanan_c': df_cuaca['suhu_rata2_c'].mean(),
    'total_curah_hujan_mm': df_cuaca['curah_hujan_mm'].sum(),
    'angin_maks_kmh': df_cuaca['kecepatan_angin_maks_kmh'].max()
}])
assert df_cuaca_ringkas['kecamatan'].is_unique, 'Kunci tabel cuaca harus unik.'

# Gabungkan sumber BPS dengan ringkasan API; many_to_one memastikan satu ringkasan cuaca per kecamatan.
df_hasil = df_akomodasi.merge(
    df_cuaca_ringkas, on='kecamatan', how='left',
    validate='many_to_one', indicator=True
)
assert len(df_hasil) == len(df_akomodasi), 'Jumlah baris berubah setelah merge.'
assert (df_hasil['_merge'] == 'both').all(), 'Ada baris yang tidak mendapat pasangan cuaca.'
df_hasil = df_hasil.drop(columns=['_merge'])
assert df_hasil.duplicated().sum() == 0, 'Ada duplikasi baris setelah merge.'
output_merge = PROCESSED_DIR/'akomodasi_cuaca_ubud.csv'
df_hasil.to_csv(output_merge, index=False, encoding='utf-8-sig')
print('Baris sebelum merge:', len(df_akomodasi))
print('Baris sesudah merge:', len(df_hasil))
print('Duplikasi sesudah merge:', int(df_hasil.duplicated().sum()))
print('Validasi merge: many_to_one | seluruh baris cocok')
display(df_hasil.head())


## Metadata provenance, hash MD5, dan ukuran file

In [ ]:
# Hitung MD5 berkas mentah dan hasil untuk catatan provenance yang dapat diverifikasi.def hitung_md5(path):    digest = hashlib.md5()    with open(path, 'rb') as file_obj:        for chunk in iter(lambda: file_obj.read(1024 * 1024), b''):            digest.update(chunk)    return digest.hexdigest()tanggal_akses = datetime.now().date().isoformat()provenance = []def tambah_provenance(nama, jenis, url, lisensi, path):    if path.exists():        provenance.append({            'sumber': nama, 'jenis': jenis, 'url': url, 'tanggal_akses': tanggal_akses,            'lisensi': lisensi, 'file_raw': str(path.relative_to(BASE_DIR)),            'ukuran_byte': path.stat().st_size, 'md5': hitung_md5(path)        })# Catat sumber BPS asli, API, robots.txt, dan HTML hanya jika file tersebut benar-benar diperoleh.tambah_provenance('BPS Kabupaten Gianyar — Direktori Hotel Kabupaten Gianyar 2017', 'Berkas PDF', 'https://gianyarkab.bps.go.id/id/publication/2017/09/25/bfada6d6856e4d0e346d8f68/direktori-hotel-kabupaten-gianyar-2017.html', 'Publikasi membatasi penggandaan/distribusi untuk tujuan komersial tanpa izin tertulis BPS; bukan lisensi terbuka tanpa syarat.', RAW_DIR/'BPS5104-direktori_hotel_kabupaten_gianyar_2017.pdf')tambah_provenance('Open-Meteo Historical Weather API', 'API JSON', api_response.url, 'CC BY 4.0; berikan atribusi Open-Meteo dan dokumentasi model.', api_raw_path)tambah_provenance('BPS Provinsi Bali robots.txt', 'Bukti kepatuhan scraping', ROBOTS_URL, 'Aturan akses crawler; bukan lisensi data.', robots_path)tambah_provenance('BPS Provinsi Bali — tabel akomodasi nonbintang 2025', 'HTML hasil scraping', SCRAPE_URL, 'Ketentuan hak cipta dan penggunaan portal BPS; atribusi wajib.', html_raw_path)pd.DataFrame(provenance).to_csv(META_DIR/'provenance.csv', index=False, encoding='utf-8-sig')# Simpan metadata hasil merge dan perbarui checklist berdasarkan bukti eksekusi aktual.metadata_merge = [{ 'file':'data/processed/akomodasi_cuaca_ubud.csv', 'jumlah_baris':len(df_hasil), 'jumlah_kolom':len(df_hasil.columns), 'kunci_merge':'kecamatan', 'validate':'many_to_one', 'duplikasi_baris':int(df_hasil.duplicated().sum()), 'baris_cocok':len(df_hasil), 'ukuran_byte':output_merge.stat().st_size, 'md5':hitung_md5(output_merge)}]pd.DataFrame(metadata_merge).to_csv(META_DIR/'metadata_hasil_penggabungan.csv', index=False, encoding='utf-8-sig')# Checklist diisi dengan status sesuai hasil pemeriksaan; tidak menyatakan scraping berhasil bila belum terbukti.check_path = META_DIR/'data_ethics_checklist.csv'if check_path.exists():    check = pd.read_csv(check_path)    if len(check) >= 9:        check.loc[2, 'Status'] = 'Ya' if robots_response is not None else 'Tidak'        check.loc[2, 'Bukti / catatan'] = f'Pemeriksaan robots.txt pada {tanggal_akses}; HTTP {robots_status}; can_fetch={allowed}; status scraping: {status_scraping}. Status Ya berarti pemeriksaan dilakukan, bukan berarti scraping pasti diizinkan.'        check.to_csv(check_path, index=False, encoding='utf-8-sig')        # Sinkronkan status yang sama ke workbook Excel yang akan dikumpulkan.        from openpyxl import load_workbook        check_xlsx = META_DIR/'data_ethics_checklist.xlsx'        if check_xlsx.exists():            wb_check = load_workbook(check_xlsx)            ws_check = wb_check['Checklist Etika']            ws_check.cell(4, 3).value = 'Ya' if allowed else 'Tidak'            ws_check.cell(4, 4).value = f'robots.txt diambil pada {tanggal_akses}; HTTP {robots_response.status_code}; can_fetch={allowed}; status scraping: {status_scraping}. robots.txt tidak menggantikan ketentuan penggunaan portal.'            wb_check.save(check_xlsx)# Simpan juga hasil provenance dan metadata untuk ditinjau sebelum pengumpulan.display(pd.DataFrame(provenance))display(pd.DataFrame(metadata_merge))print('Status scraping:', status_scraping)

## Kesimpulan

In [ ]:
# Ringkas hasil akhir dan tampilkan secara transparan bagian yang berhasil atau masih memerlukan tindakan.
print(f'Dataset akomodasi: {len(df_akomodasi)} baris dan {len(df_akomodasi.columns)} kolom.')
print(f'Data API: {len(df_cuaca)} observasi harian.')
print(f'Tabel hasil scraping: {len(tables)} tabel ditemukan.')
print(f'Hasil merge: {len(df_hasil)} baris; duplikasi: {df_hasil.duplicated().sum()}.')
print('Validasi merge: many_to_one; seluruh baris memiliki pasangan cuaca.')
print('Status scraping:', status_scraping)
print('Catatan: direktori akomodasi tahun 2015; cuaca Januari 2025; tabel scraping BPS merupakan statistik agregat.')
print('Periksa status scraping. Jika robots melarang atau tabel statis tidak ditemukan, bagian scraping belum terpenuhi dan perlu konfirmasi dosen/sumber lain yang diizinkan.')
print('Simpan notebook setelah semua sel dijalankan dan periksa folder data/raw, data/processed, dan metadata.')
